# Notebook 1: LLM Fundamentals with LangChain

**Learning Goal**: Understand what LangChain is, set up your environment, make your first LLM call, and learn about prompts, messages, and output parsing.

**rag_engine.py coverage**: This notebook covers the foundational building blocks used throughout the production script — the LLM, prompt templates, message types, and output parsers.

---

## 1. What is LangChain?

LangChain is an open-source framework for building applications powered by Large Language Models (LLMs). Instead of writing raw API calls, LangChain provides:

- **Standardized interfaces** — swap between OpenAI, Anthropic, or local models with the same code
- **Composable chains** — connect components together using the pipe (`|`) operator
- **Built-in integrations** — document loaders, vector stores, retrievers, and more

### Key Abstractions

| Concept | What it does | Example |
|---|---|---|
| **Chat Model** | Sends messages to an LLM and gets a response | `ChatOpenAI` |
| **Prompt Template** | Formats inputs into a structured prompt | `ChatPromptTemplate` |
| **Output Parser** | Extracts usable data from the LLM response | `StrOutputParser` |
| **Chain** | Connects components: `prompt \| llm \| parser` | LCEL chain |
| **Retriever** | Fetches relevant documents from a knowledge base | FAISS retriever |

Our production script `rag_engine.py` uses all of these. In this notebook, we'll learn the first four.

## 2. Environment Setup

Before we can use LangChain with OpenAI, we need:
1. The required Python packages
2. An OpenAI API key stored in a `.env` file

### Installing Dependencies

Run the cell below to install the packages (skip if already installed):

In [ ]:
# Install required packages
# %pip install langchain langchain-openai python-dotenv

### Loading Your API Key

We use `python-dotenv` to load environment variables from a `.env` file. This keeps your API key out of your code.

Create a `.env` file in the project root with:
```
OPENAI_API_KEY=sk-your-key-here
```

> **In `rag_engine.py`** (line 10-12): `load_dotenv()` is called at the top of the script to make the API key available.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify the key is loaded (prints first 8 chars only)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API key loaded: {api_key[:8]}...")
else:
    print("WARNING: No OPENAI_API_KEY found. Create a .env file in the project root.")

## 3. Your First LLM Call

The `ChatOpenAI` class is LangChain's wrapper around OpenAI's chat models. Key parameters:

- **`model`**: Which model to use (e.g., `"gpt-4o"`, `"gpt-4o-mini"`)
- **`temperature`**: Controls randomness (0 = deterministic, 1 = creative)

> **In `rag_engine.py`** (line 59): `ChatOpenAI(model="gpt-4o", temperature=0)` — temperature=0 ensures consistent, factual answers for the FAQ bot.

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the LLM — same as rag_engine.py line 59
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Make a simple call
response = llm.invoke("What is RAG in the context of AI?")
print(response)

Notice that the response is an `AIMessage` object, not a plain string. It contains:
- `content`: The actual text response
- `response_metadata`: Token usage, model info, etc.

Let's access just the content:

In [ ]:
# The response is an AIMessage object
print(f"Type: {type(response)}")
print(f"Content: {response.content}")

## 4. Messages & Roles

Chat models work with a **list of messages**, where each message has a **role**:

| Role | Purpose | LangChain Class |
|---|---|---|
| **system** | Sets the AI's behavior and persona | `SystemMessage` |
| **human** | The user's input | `HumanMessage` |
| **assistant** / **ai** | The AI's previous responses | `AIMessage` |

This message format is how we give the LLM context about the conversation.

> **In `rag_engine.py`** (line 9): `AIMessage` and `HumanMessage` are imported to represent chat history.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# A conversation is a list of messages
messages = [
    SystemMessage(content="You are a helpful healthcare assistant. Answer in one sentence."),
    HumanMessage(content="What is health insurance?"),
]

response = llm.invoke(messages)
print(response.content)

### Multi-turn Conversation

To simulate a conversation with history, we include previous exchanges in the message list. This is exactly how `rag_engine.py` handles chat history — by passing a list of `HumanMessage` and `AIMessage` objects.

In [ ]:
# Multi-turn conversation: the LLM sees the full history
messages = [
    SystemMessage(content="You are a helpful healthcare assistant. Answer in one sentence."),
    HumanMessage(content="What is health insurance?"),
    AIMessage(content="Health insurance is a financial product that covers medical expenses in exchange for regular premium payments."),
    HumanMessage(content="Why is it important?"),  # Follow-up question
]

response = llm.invoke(messages)
print(response.content)

## 5. Prompt Templates

Hardcoding messages is not scalable. **Prompt templates** let you define reusable prompt structures with **variables** that get filled in at runtime.

### ChatPromptTemplate

`ChatPromptTemplate.from_messages()` creates a template from a list of `(role, content)` tuples. Variables are indicated with `{curly_braces}`.

> **In `rag_engine.py`** (lines 70-76): The contextualization prompt uses a template with `{input}` variable and `MessagesPlaceholder("chat_history")` for dynamic history.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert on {topic}. Answer in one sentence."),
    ("human", "{question}"),
])

# Format the template with actual values
formatted = prompt.invoke({"topic": "health insurance", "question": "What is NHIF?"})
print("Formatted messages:")
for msg in formatted.messages:
    print(f"  [{msg.type}]: {msg.content}")

### MessagesPlaceholder

When you need to insert a **variable-length list of messages** (like chat history), use `MessagesPlaceholder`. It acts as a slot where multiple messages can be injected.

> **In `rag_engine.py`** (line 73, 94): `MessagesPlaceholder("chat_history")` is used in both the contextualization and QA prompts to inject conversation history.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

# A prompt that accepts dynamic chat history
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in one sentence."),
    MessagesPlaceholder("chat_history"),  # <-- dynamic message list goes here
    ("human", "{input}"),
])

# Provide chat history as a list of messages
chat_history = [
    HumanMessage(content="What is NHIF?"),
    AIMessage(content="NHIF is Tanzania's National Health Insurance Fund."),
]

formatted = prompt_with_history.invoke({
    "chat_history": chat_history,
    "input": "When was it established?"
})

print("Formatted messages:")
for msg in formatted.messages:
    print(f"  [{msg.type}]: {msg.content}")

## 6. Output Parsers

LLM responses are `AIMessage` objects. Often, we just want the **string content**. That's what `StrOutputParser` does — it extracts `.content` from the response.

### Why use a parser instead of `.content`?

Because parsers are **composable** — they can be chained with other components using the `|` (pipe) operator. This is the foundation of LangChain Expression Language (LCEL).

> **In `rag_engine.py`** (line 78): `StrOutputParser()` is chained at the end of the contextualization chain.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Without parser: returns an AIMessage object
raw_response = llm.invoke("Say hello")
print(f"Without parser: {type(raw_response)} -> {raw_response}")
print()

# With parser: returns a plain string
parser = StrOutputParser()
parsed = parser.invoke(raw_response)
print(f"With parser: {type(parsed)} -> {parsed}")

### The Pipe Operator (`|`)

LangChain's most powerful feature is **chaining** components with the `|` operator. Data flows left to right:

```
prompt | llm | parser
```

This means:
1. Format the input using the prompt template
2. Send the formatted messages to the LLM
3. Parse the LLM response into a string

The result is a **chain** — a single callable that takes input and returns output.

In [ ]:
# Build a chain: prompt -> llm -> parser
chain = prompt_with_history | llm | StrOutputParser()

# Invoke the chain with all required variables
result = chain.invoke({
    "chat_history": [],  # empty history for first message
    "input": "What is health insurance?"
})

print(f"Type: {type(result)}")  # str, not AIMessage!
print(f"Result: {result}")

## 7. Putting It Together: The Contextualization Chain

Let's rebuild the **contextualization chain** from `rag_engine.py` (lines 62-78). This chain takes a follow-up question and rewrites it as a standalone question.

**Why is this needed?** In a RAG system, we search for relevant documents using the user's question. But follow-up questions like *"How much does it cost?"* don't make sense without knowing what *"it"* refers to. The contextualization chain resolves these references.

### Production Code Reference
```python
# rag_engine.py lines 62-78
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

cntx_q_chain = contextualize_q_prompt | llm | StrOutputParser()
```

In [ ]:
# Rebuild the contextualization chain from rag_engine.py

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Chain: prompt -> llm -> parser
cntx_q_chain = contextualize_q_prompt | llm | StrOutputParser()

print("Chain created:", type(cntx_q_chain))

In [ ]:
# Test: a follow-up question that references chat history
chat_history = [
    HumanMessage(content="What is NHIF?"),
    AIMessage(content="NHIF is Tanzania's National Health Insurance Fund."),
]

# "it" refers to NHIF from the history
standalone = cntx_q_chain.invoke({
    "chat_history": chat_history,
    "input": "How much does it cost?"
})

print(f"Original: 'How much does it cost?'")
print(f"Standalone: '{standalone}'")

In [ ]:
# Test: a question that doesn't need rewriting
standalone = cntx_q_chain.invoke({
    "chat_history": [],
    "input": "What are the NHIF benefit packages?"
})

print(f"Original: 'What are the NHIF benefit packages?'")
print(f"Standalone: '{standalone}'")
# Should return the question as-is since there's no history to resolve

## Summary

In this notebook we learned:

| Concept | What we learned | rag_engine.py reference |
|---|---|---|
| `ChatOpenAI` | Initialize and call an LLM | Line 59 |
| `HumanMessage` / `AIMessage` | Represent conversation messages | Line 9 |
| `ChatPromptTemplate` | Create reusable prompt structures | Lines 70-76, 91-97 |
| `MessagesPlaceholder` | Inject dynamic chat history | Lines 73, 94 |
| `StrOutputParser` | Extract string from LLM response | Line 78 |
| Pipe operator `\|` | Chain components together | Line 78 |

**Next notebook**: We'll learn how to load documents, split them into chunks, create embeddings, and store them in a vector database (FAISS).